In [1]:
import pandas as pd

transaction = pd.read_excel("../data/cleaned/Transaction_cleaned.xlsx")
visit_mode = pd.read_excel("../data/cleaned/Mode_cleaned.xlsx")

print(transaction.shape)
print(transaction.head())

(52930, 7)
   TransactionId  UserId  VisitYear  VisitMonth  VisitMode  AttractionId  \
0              3   70456       2022          10          2           640   
1              8    7567       2022          10          4           640   
2              9   79069       2022          10          3           640   
3             10   31019       2022          10          3           640   
4             15   43611       2022          10          2           640   

   Rating  
0       5  
1       5  
2       5  
3       3  
4       3  


In [2]:
# Merge Transaction with Mode, matching VisitMode (in transaction) to VisitModeId (in mode table)
master = transaction.merge(visit_mode, left_on='VisitMode', right_on='VisitModeId', how='left')

# Check the result
print(master.shape)
print(master[['TransactionId', 'VisitMode_x', 'VisitMode_y']].head())

(52930, 9)
   TransactionId  VisitMode_x VisitMode_y
0              3            2     Couples
1              8            4     Friends
2              9            3      Family
3             10            3      Family
4             15            2     Couples


# Clean up the column names

In [3]:
# Drop the redundant numeric VisitMode column and the duplicate ID column, keep the readable label
master = master.drop(columns=['VisitMode_x', 'VisitModeId'])

# Rename VisitMode_y back to a clean name
master = master.rename(columns={'VisitMode_y': 'VisitMode'})

# Check result
print(master.shape)
print(master.columns.tolist())
print(master.head())

(52930, 7)
['TransactionId', 'UserId', 'VisitYear', 'VisitMonth', 'AttractionId', 'Rating', 'VisitMode']
   TransactionId  UserId  VisitYear  VisitMonth  AttractionId  Rating  \
0              3   70456       2022          10           640       5   
1              8    7567       2022          10           640       5   
2              9   79069       2022          10           640       5   
3             10   31019       2022          10           640       3   
4             15   43611       2022          10           640       3   

  VisitMode  
0   Couples  
1   Friends  
2    Family  
3    Family  
4   Couples  


# Merge in Attraction (Item) details

In [4]:
item = pd.read_excel("../data/cleaned/Item_cleaned.xlsx")

# Merge master with item, matching on AttractionId (same column name in both, so this is simpler)
master = master.merge(item, on='AttractionId', how='left')

print(master.shape)
print(master.columns.tolist())
print(master.head())

(52930, 11)
['TransactionId', 'UserId', 'VisitYear', 'VisitMonth', 'AttractionId', 'Rating', 'VisitMode', 'AttractionCityId', 'AttractionTypeId', 'Attraction', 'AttractionAddress']
   TransactionId  UserId  VisitYear  VisitMonth  AttractionId  Rating  \
0              3   70456       2022          10           640       5   
1              8    7567       2022          10           640       5   
2              9   79069       2022          10           640       5   
3             10   31019       2022          10           640       3   
4             15   43611       2022          10           640       3   

  VisitMode  AttractionCityId  AttractionTypeId  \
0   Couples                 1                63   
1   Friends                 1                63   
2    Family                 1                63   
3    Family                 1                63   
4   Couples                 1                63   

                       Attraction                        AttractionAddres

# Merge in Attraction Type name

In [5]:
attraction_type = pd.read_excel("../data/cleaned/Type_cleaned.xlsx")

master = master.merge(attraction_type, on='AttractionTypeId', how='left')

print(master.shape)
print(master[['Attraction', 'AttractionTypeId', 'AttractionType']].head())

(52930, 12)
                       Attraction  AttractionTypeId           AttractionType
0  Sacred Monkey Forest Sanctuary                63  Nature & Wildlife Areas
1  Sacred Monkey Forest Sanctuary                63  Nature & Wildlife Areas
2  Sacred Monkey Forest Sanctuary                63  Nature & Wildlife Areas
3  Sacred Monkey Forest Sanctuary                63  Nature & Wildlife Areas
4  Sacred Monkey Forest Sanctuary                63  Nature & Wildlife Areas


# Merge in User demographics

In [6]:
user = pd.read_excel("../data/cleaned/User_cleaned.xlsx")

master = master.merge(user, on='UserId', how='left')

print(master.shape)
print(master.columns.tolist())

(52930, 16)
['TransactionId', 'UserId', 'VisitYear', 'VisitMonth', 'AttractionId', 'Rating', 'VisitMode', 'AttractionCityId', 'AttractionTypeId', 'Attraction', 'AttractionAddress', 'AttractionType', 'ContinentId', 'RegionId', 'CountryId', 'CityId']


# Merge in readable names for Continent, Region, Country, City

In [7]:
continent = pd.read_excel("../data/cleaned/Continent_cleaned.xlsx")
region = pd.read_excel("../data/cleaned/Region_cleaned.xlsx")
country = pd.read_excel("../data/cleaned/Country_cleaned.xlsx")
city = pd.read_excel("../data/cleaned/City_cleaned.xlsx")

# Merge Continent name
master = master.merge(continent, on='ContinentId', how='left')

# Merge Region name (drop the duplicate ContinentId that comes along with the Region table)
master = master.merge(region, on='RegionId', how='left', suffixes=('', '_region'))
master = master.drop(columns=['ContinentId_region'])

# Merge Country name (drop the duplicate RegionId that comes along with the Country table)
master = master.merge(country, on='CountryId', how='left', suffixes=('', '_country'))
master = master.drop(columns=['RegionId_country'])

# Merge City name (drop the duplicate CountryId that comes along with the City table)
master = master.merge(city, on='CityId', how='left', suffixes=('', '_city'))
master = master.drop(columns=['CountryId_city'])

print(master.shape)
print(master.columns.tolist())

(52930, 20)
['TransactionId', 'UserId', 'VisitYear', 'VisitMonth', 'AttractionId', 'Rating', 'VisitMode', 'AttractionCityId', 'AttractionTypeId', 'Attraction', 'AttractionAddress', 'AttractionType', 'ContinentId', 'RegionId', 'CountryId', 'CityId', 'Continent', 'Region', 'Country', 'CityName']


# Final sanity check before saving

In [8]:
# Check for any new missing values introduced by merging
print(master.isnull().sum())

# Quick look at a few random rows to eyeball that everything lines up correctly
print(master.sample(5))

TransactionId        0
UserId               0
VisitYear            0
VisitMonth           0
AttractionId         0
Rating               0
VisitMode            0
AttractionCityId     0
AttractionTypeId     0
Attraction           0
AttractionAddress    0
AttractionType       0
ContinentId          0
RegionId             0
CountryId            0
CityId               0
Continent            0
Region               0
Country              0
CityName             0
dtype: int64
       TransactionId  UserId  VisitYear  VisitMonth  AttractionId  Rating  \
10232          14760   33287       2014           7           640       4   
16237          24667   86538       2016           8           841       4   
20923          33232   49036       2017           5           673       3   
16637          25245   44835       2017           5           841       3   
38256         100401   80946       2017          10           749       3   

      VisitMode  AttractionCityId  AttractionTypeId  \
10232   F

# Save the master dataset

In [9]:
master.to_excel("../data/cleaned/master_dataset.xlsx", index=False)
print("Saved! Final shape:", master.shape)

Saved! Final shape: (52930, 20)
